# STL decision trees: naval quickstart

This notebook uses the bundled real naval dataset. It is an onboarding run (12 original trajectories, depth 1), not reproduction of all paper experiments. All 1,891 intervals are retained. Run it in the installed project environment on a remote lab with a suitable Gurobi license. See `docs/semantics.md` for inherited bounds and equality behavior.

## 1. Locate the repository and load the data

Run from the repository root or the `examples` directory. Pickle input is explicitly trusted; prefer numeric NPZ for new data.

In [ ]:
from pathlib import Path

import numpy as np

from stl_tree import (
    ModelConfig,
    SolverConfig,
    augment_with_discrete_feature,
    build_model,
    load_dataset,
    sample_data,
    solve,
)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/naval_dataset.pkl").exists())
dataset = load_dataset(ROOT / "data/naval_dataset.pkl", trusted_pickle=True)
dataset.summary()

## 2. Inspect a trajectory

Channel names and sample indices are used because physical units were not supplied with the attachment.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
for feature, values in enumerate(dataset.train_signals[0]):
    ax.plot(values, label=f"s{feature + 1}")
ax.set(xlabel="Time-step index", ylabel="Signal value", title="One real naval trajectory")
ax.legend()
fig.tight_layout()
plt.show()

## 3. Sample and apply the original augmentation

Only the training split is sampled. The recipe appends a zero-valued binary channel, copies half the label-1 trajectories, sets the copies' channel to one, and relabels them zero. This changes the experimental classification problem; it is not generic preprocessing.

In [ ]:
X, y, selected = sample_data(*dataset.select("train"), samples=12, seed=42)
X, y, parents = augment_with_discrete_feature(X, y)
print("Augmented signals:", X.shape)
print("Source indices:", selected[parents])

## 4. Build and solve

The solver has 30 seconds and one thread. A feasible incumbent may be returned before the optimum is proved. The `finally` block releases solver resources.

In [ ]:
artifacts = build_model(X, y, ModelConfig(depth=1, discrete_mode="exist"))
try:
    print("Temporal windows:", len(artifacts.windows))
    result = solve(artifacts, SolverConfig(time_limit=30, threads=1))
finally:
    artifacts.model.dispose()
result.to_dict()

## 5. Inspect the tree and evaluate on the held-out split

The intended setting excludes zero robustness. This small depth-1 example can nevertheless return an equality optimum: inspect result.threshold_stabilization before interpreting its flow score as strict classification accuracy. Export preserves stable flow where possible and reports unresolved boundaries. Prediction below explicitly chooses left and reports ties. We independently apply the same experimental augmentation within the test split. This is valid held-out evaluation only because the model above uses the training split, not the pooled naval preset.

In [ ]:
if result.tree is not None:
    print(result.tree.format())
    X_test, y_test, test_parents = augment_with_discrete_feature(*dataset.select("test"))
    predictions, ties = result.tree.predict(X_test, tie_break="left", return_ties=True)
    print("Augmented held-out accuracy:", float(np.mean(predictions == y_test)))
    print("Prediction ties:", int(ties.sum()))
else:
    print("No incumbent:", result.status)

## 6. Save the run

Use a fresh directory to keep each run's configuration and output together. A tree is saved only if a feasible incumbent exists.

In [ ]:
from datetime import datetime

output = ROOT / "outputs" / ("notebook-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
result.save(
    output,
    metadata={
        "split": "train",
        "augmentation": "discrete",
        "sampling_seed": 42,
        "source_indices": selected[parents].tolist(),
    },
)
print(output)